In [0]:
%run "../../commons/commons_imports"

In [0]:
df_estado_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_ESTADO,
    format="delta"
)

In [0]:

window = Window.partitionBy("CO_MUNICIPIO").orderBy(col("ANO_REFERENCIA").desc())

df_metas_municipio = (
    read(
        base_path=SILVER_PATH,
        table_name=METAS_MUNICIPIO,
        format="delta"
    )
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
).select(
    'CO_MUNICIPIO',
    'META_ALFABETIZACAO_2024',
    'META_ALFABETIZACAO_2025',
    'META_ALFABETIZACAO_2026',
    'META_ALFABETIZACAO_2027',
    'META_ALFABETIZACAO_2028',
    'META_ALFABETIZACAO_2029',
    'META_ALFABETIZACAO_2030',
)


In [0]:
window = Window.partitionBy("SG_UF").orderBy(col("ANO_REFERENCIA").desc())

df_metas_uf = (
    read(
        base_path=SILVER_PATH,
        table_name=METAS_UF,
        format="delta"
    )
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
).select(
    'SG_UF',
    'META_ALFABETIZACAO_2024',
    'META_ALFABETIZACAO_2025',
    'META_ALFABETIZACAO_2026',
    'META_ALFABETIZACAO_2027',
    'META_ALFABETIZACAO_2028',
    'META_ALFABETIZACAO_2029',
    'META_ALFABETIZACAO_2030',
)

In [0]:
df_metas_br = read(
    base_path=SILVER_PATH,
    table_name=METAS_BR,
    format="delta"
).withColumn(
    'TIPO DE META',lit('BRASIL')
).select(
    'TIPO DE META',
    'META_ALFABETIZACAO_2024',
    'META_ALFABETIZACAO_2025',
    'META_ALFABETIZACAO_2026',
    'META_ALFABETIZACAO_2027',
    'META_ALFABETIZACAO_2028',
    'META_ALFABETIZACAO_2029',
    'META_ALFABETIZACAO_2030',
).dropDuplicates(['TIPO DE META'])

In [0]:
from pyspark.sql.functions import col, expr

df_pivot_metas_br = (
    df_metas_br
    .select(
        col("TIPO DE META").alias("TIPO_META"),
        expr("""
            stack(
                7,
                2024, META_ALFABETIZACAO_2024,
                2025, META_ALFABETIZACAO_2025,
                2026, META_ALFABETIZACAO_2026,
                2027, META_ALFABETIZACAO_2027,
                2028, META_ALFABETIZACAO_2028,
                2029, META_ALFABETIZACAO_2029,
                2030, META_ALFABETIZACAO_2030
            ) AS (ANO_META, META_ALFABETIZACAO_BRASIL)
        """)
    )
)


df_pivot_metas_uf = (
    df_metas_uf
    .select(
        col("SG_UF").alias("SG_UF_META"),
        expr("""
            stack(
                7,
                2024, META_ALFABETIZACAO_2024,
                2025, META_ALFABETIZACAO_2025,
                2026, META_ALFABETIZACAO_2026,
                2027, META_ALFABETIZACAO_2027,
                2028, META_ALFABETIZACAO_2028,
                2029, META_ALFABETIZACAO_2029,
                2030, META_ALFABETIZACAO_2030
            ) AS (ANO_META, META_ALFABETIZACAO_UF)
        """)
    
    )
)


df_pivot_metas_municipio = (
    df_metas_municipio
    .select(
        col("CO_MUNICIPIO").alias("CO_MUNICIPIO_META"),
        expr("""
            stack(
                7,
                2024, META_ALFABETIZACAO_2024,
                2025, META_ALFABETIZACAO_2025,
                2026, META_ALFABETIZACAO_2026,
                2027, META_ALFABETIZACAO_2027,
                2028, META_ALFABETIZACAO_2028,
                2029, META_ALFABETIZACAO_2029,
                2030, META_ALFABETIZACAO_2030
            ) AS (ANO_META, META_ALFABETIZACAO_MUNICIPIO)
        """)
    )
)

In [0]:
df_estado_indicador = (
    df_estado_silver
    .select(
        "ANO_REFERENCIA",
        "CO_UF",
        "ID_TIPO_REDE",
        col("PC_ALUNO_ALFABETIZADO")
            .alias("PC_ALUNO_ALFABETIZADO_ESTADO"),
        col("VL_MEDIA_LP")
            .alias("VL_MEDIA_LP_ESTADO")
    )

)

In [0]:
df_municipio_silver = read(
    base_path=SILVER_PATH,
    table_name=TS_MUNICIPIO,
    format="delta"
).select(
    col("ANO_REFERENCIA"),
    col("CO_UF"),
    col("SG_UF"),
    col("REGIAO"),
    col("CO_MUNICIPIO"),
    col("NO_MUNICIPIO"),
    col("NO_MUNICIPIO_UF"),
    col("ID_TIPO_REDE"),
    col("DS_TIPO_REDE"),
    col("PC_ALUNO_ALFABETIZADO"),
    col("VL_MEDIA_LP"),
    col("FAIXA_MEDIA_LP"),
    col("FAIXA_ALFABETIZACAO")
).filter(~col('ID_TIPO_REDE').isin([0,5]))

In [0]:
### meta br

df_municipio_silver_metas_br = df_municipio_silver.join(
    df_pivot_metas_br,
    how='left',
    on=[df_municipio_silver.ANO_REFERENCIA == df_pivot_metas_br.ANO_META]
).drop(
    'TIPO_META',
    'ANO_META'
)
# ### meta municipio
df_municipio_silver_metas_uf = df_municipio_silver_metas_br.join(
    df_pivot_metas_uf,
    how='left',
    on=[(df_municipio_silver_metas_br.ANO_REFERENCIA == df_pivot_metas_uf.ANO_META) & (df_municipio_silver_metas_br.SG_UF == df_pivot_metas_uf.SG_UF_META)]
).drop(
    'SG_UF_META',
    'ANO_META'
)

# ##3 meta estado
df_municipio_silver_metas_municipio = df_municipio_silver_metas_uf.join(
    df_pivot_metas_municipio,
    how='left',
    on=[(df_municipio_silver_metas_uf.ANO_REFERENCIA == df_pivot_metas_municipio.ANO_META) & (df_municipio_silver_metas_uf.CO_MUNICIPIO
 == df_pivot_metas_municipio.CO_MUNICIPIO_META)]
).drop(
    'CO_MUNICIPIO_META',
    'ANO_META'
)

In [0]:
df_indicador_municipio = (

    df_municipio_silver_metas_municipio

    # =====================================================
    # Diferença para a Meta
    # =====================================================

    .withColumn(
        "DIF_META_ALFABETIZACAO_BRASIL",
        round(
            col("PC_ALUNO_ALFABETIZADO") - col("META_ALFABETIZACAO_BRASIL"),
            2
        )
    )

    .withColumn(
        "DIF_META_ALFABETIZACAO_UF",
        round(
            col("PC_ALUNO_ALFABETIZADO") - col("META_ALFABETIZACAO_UF"),
            2
        )
    )

    .withColumn(
        "DIF_META_ALFABETIZACAO_MUNICIPIO",
        round(
            col("PC_ALUNO_ALFABETIZADO") - col("META_ALFABETIZACAO_MUNICIPIO"),
            2
        )
    )

    # =====================================================
    # Atingiu a Meta
    # =====================================================

    .withColumn(
        "ATINGIU_META_BRASIL",
        when(
            col("PC_ALUNO_ALFABETIZADO") >= col("META_ALFABETIZACAO_BRASIL"),
            "Sim"
        ).otherwise("Não")
    )

    .withColumn(
        "ATINGIU_META_UF",
        when(
            col("PC_ALUNO_ALFABETIZADO") >= col("META_ALFABETIZACAO_UF"),
            "Sim"
        ).otherwise("Não")
    )

    .withColumn(
        "ATINGIU_META_MUNICIPIO",
        when(
            col("PC_ALUNO_ALFABETIZADO") >= col("META_ALFABETIZACAO_MUNICIPIO"),
            "Sim"
        ).otherwise("Não")
    )

    # =====================================================
    # Classificação do Município
    # =====================================================

    .withColumn(
        "CLASSIFICACAO",
        when(col("PC_ALUNO_ALFABETIZADO") >= 80, "Nível 5")
        .when(col("PC_ALUNO_ALFABETIZADO") >= 70, "Nível 4")
        .when(col("PC_ALUNO_ALFABETIZADO") >= 60, "Nível 3")
        .when(col("PC_ALUNO_ALFABETIZADO") >= 50, "Nível 2")
        .when(col("PC_ALUNO_ALFABETIZADO") >= 40, "Nível 1")
        .otherwise("Abaixo do Nível 1")
    )

    # =====================================================
    # Ordem da Classificação
    # Facilita ordenação no Power BI
    # =====================================================

    .withColumn(
        "ORDEM_CLASSIFICACAO",
        when(col("CLASSIFICACAO") == "Nível 5", 5)
        .when(col("CLASSIFICACAO") == "Nível 4", 4)
        .when(col("CLASSIFICACAO") == "Nível 3", 3)
        .when(col("CLASSIFICACAO") == "Nível 2", 2)
        .when(col("CLASSIFICACAO") == "Nível 1", 1)
        .otherwise(0)
    )

)

In [0]:
df_indicador_municipio_joined = (

    df_indicador_municipio.alias("m")

    .join(
        df_estado_indicador.alias("e"),
        on=[
            col("m.ANO_REFERENCIA") == col("e.ANO_REFERENCIA"),
            col("m.CO_UF") == col("e.CO_UF"),
            col("m.ID_TIPO_REDE") == col("e.ID_TIPO_REDE")
        ],
        how="left"
    )
    .drop(
        col("e.ANO_REFERENCIA"),
        col("e.CO_UF"),
        col("e.ID_TIPO_REDE")
    )

)

In [0]:
df_indicador_municipio_joined = (
    df_indicador_municipio_joined
    .withColumn(
        "DIF_ALFABETIZACAO_ESTADO",
        round(
            col("PC_ALUNO_ALFABETIZADO")
            -
            col("PC_ALUNO_ALFABETIZADO_ESTADO"),
            2
        )

    )
    .withColumn(
        "DIF_MEDIA_LP_ESTADO",
        round(
            col("VL_MEDIA_LP")
            -
            col("VL_MEDIA_LP_ESTADO"),
            2
        )

    )
    .withColumn(
        "ACIMA_MEDIA_ESTADO",
        when(
            col("DIF_ALFABETIZACAO_ESTADO") >= 0,
            "Sim"
        ).otherwise("Não")

    )

)

In [0]:
window_municipio = (
    Window
    .partitionBy(
        "CO_MUNICIPIO",
        "ID_TIPO_REDE"
    )
    .orderBy(
        "ANO_REFERENCIA"
    )

)

In [0]:
df_indicador_municipio_joined = (

    df_indicador_municipio_joined
    .withColumn(
        "VARIACAO_ALFABETIZACAO",
        round(
            col("PC_ALUNO_ALFABETIZADO")
            -
            lag(

                "PC_ALUNO_ALFABETIZADO"
            ).over(window_municipio),

            2
        )
    )

    .withColumn(
        "VARIACAO_MEDIA_LP",
        round(
            col("VL_MEDIA_LP")
            -
            lag(
                "VL_MEDIA_LP"
            ).over(window_municipio),
            2
        )
    )
)

In [0]:
df_indicador_municipio_joined = (
    df_indicador_municipio_joined
    .withColumn(
        "TENDENCIA",
        when(
            col("VARIACAO_ALFABETIZACAO") > 0,
            "Melhorou"
        )
        .when(
            col("VARIACAO_ALFABETIZACAO") < 0,
            "Piorou"
        )
        .when(
            col("VARIACAO_ALFABETIZACAO") == 0,
            "Estável"
        )
    )
)

In [0]:
write_delta(
    df=df_indicador_municipio_joined,
    base_path=GOLD_PATH,
    table_name=INDICADOR_MUNICIPIO,
    write_mode="overwrite"
)